In [1]:
import numpy as np
import yfinance as fy
import QuantLib as ql
from typing import Optional, Union
import pandas as pd
import matplotlib.pyplot as plt
import mocaxpy
import multiprocess
import time
import seaborn as sb

from inception.instruments.constant_parameters import NOTES_PARAMETERS
from inception.utils import timer
from inception.quantlib_python import date_to_date
from inception.instruments import ReferenceMultiIndexReturnSimulator, MultiAssetAutoCallableNote
from inception.utils import timer
from tqdm import tqdm

In [2]:
def autocallable_note_pricer(spot_price: list, evaluate_date: Union[str, pd.Timestamp]):
    """
    The Autocallable Notes pricer function wrapper, (adapts the original function signature to the signature expected by MoCaX)
    :param spot_price: [stock A price, stock B price, stock C price]
    :param evaluate_date: 
    """
    import numpy as np
    import pandas as pd
    from inception.instruments.constant_parameters import NOTES_PARAMETERS
    from inception.instruments import ReferenceMultiIndexReturnSimulator, MultiAssetAutoCallableNote
    
    start_date = '2024-02-14'
    corr_matrix = np.array(
        [[1.        , 0.96432854, 0.80799003],
        [0.96432854, 1.        , 0.8267037 ],
        [0.80799003, 0.8267037 , 1.        ]]
    )
    issued_price = [382.82, 494.08, 142.86]
    index_vol = [0.16, 0.16, 0.16]
    risk_free_rate = 0.05
    path_num = 100_000
    d_s = 0.02
    parameters = NOTES_PARAMETERS['Multi Assets One Year']
    simulator = ReferenceMultiIndexReturnSimulator(issued_price, spot_price, risk_free_rate, index_vol, corr_matrix, d_s)
    notes = MultiAssetAutoCallableNote(evaluate_date,
                                       ['A', 'B', 'C'],
                                       simulator, 
                                       parameters['valuation dates'],
                                       parameters['coupon dates'],
                                       parameters['call dates'],
                                       coupon_barrier=parameters['coupon barrier'],
                                       call_barrier=parameters['call barrier'],
                                       principal_barrier=parameters['principal barrier'],
                                       coupon_amount=parameters['coupon amount'],
                                       free_rate=risk_free_rate,
                                       n_paths=path_num)

    
    return notes.value

In [3]:
autocallable_note_pricer([382.82, 494.08, 142.86], '2024-04-15')

102.04206522075667

In [4]:
@timer
def autocallable_notes_chebyshev_approximate(evaluate_date: Union[str, pd.Timestamp], pricer: 'function'):
    """
    Using Chebyshev Tensor to approximate the Autocallable notes pricer
    :param evaluate_date: 
    :return:
    """
    import pandas as pd
    import mocaxpy
    import numpy as np
    import os
    # Number of dimensions
    num_dimensions = 3

    # Function domain for stock A, B and C price, Time to Maturity
    issued_price = [382.82, 494.08, 142.86]

    lower_bound = 0.2
    upper_bound = 1.1

    domain_values = [
        [issued_price[0] * lower_bound, issued_price[0] * upper_bound] ,  # Stock A price
        [issued_price[1] * lower_bound, issued_price[1] * upper_bound] ,  # Stock B price
        [issued_price[2] * lower_bound, issued_price[2] * upper_bound] ,  # Stock C price
    ]
    domain = mocaxpy.MocaxDomain(domain_values)

    # MoCaX accuracy parameters, Chebyshev Nodes
    n_nodes = [9, 9, 9]
    mocax_nodes = mocaxpy.MocaxNs(n_nodes)

    # Maximum derivative order.
    max_derivative_order = 2

    option_mocax = mocaxpy.Mocax(None, 
                                 num_dimensions,
                                 domain, 
                                 None, 
                                 mocax_nodes,
                                 max_derivative_order=max_derivative_order)
    # Get the Chebysheve points
    chebysheve_points = option_mocax.get_evaluation_points()
    
    y = [pricer(x, evaluate_date) for x in chebysheve_points]
    # Set the y to Chebysheve object
    option_mocax.set_original_function_values(y)
    
    file_dir = f'./one_year_chebyshev_database/'
    if not os.path.exists(file_dir):
        os.makedirs(file_dir)
        
    file_name = file_dir + f'{evaluate_date}.mcx'
    option_mocax.serialize(file_name)
    
    
def create_chebyshev_approximator(start_date: str, end_date: str, chebyshev_approximator: 'function', pricer: 'function'):
    """
    Create chebyshev object daily
    """
    import pandas as pd
    from tqdm import tqdm
    
    date_range = [d.strftime('%Y-%m-%d') for d in pd.date_range(start_date, end_date)]
    for d in tqdm(date_range):
        chebyshev_approximator(d, pricer)   

In [5]:
# create_chebyshev_approximator('2024-02-14', '2024-03-14',
#                               autocallable_notes_chebyshev_approximate,
#                               autocallable_note_pricer)

In [6]:
#autocallable_notes_chebyshev_approximate('2024-05-15', autocallable_note_pricer)

In [9]:
## Anil

# Please run three times, and remove the comments in different intervals each time
# --------------------------------------------------
# time_bucket_1 = ('2024-02-14', '2024-03-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
# time_bucket_2 = ('2024-03-15', '2024-04-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
# time_bucket_3 = ('2024-04-15', '2024-05-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
# time_bucket_4 = ('2024-05-15', '2024-06-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)

# time_bucket_1 = ('2024-06-15', '2024-07-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
# time_bucket_2 = ('2024-07-15', '2024-08-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
# time_bucket_3 = ('2024-08-15', '2024-09-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
# time_bucket_4 = ('2024-09-15', '2024-10-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)

time_bucket_1 = ('2024-10-15', '2024-11-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
time_bucket_2 = ('2024-11-15', '2024-12-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
time_bucket_3 = ('2024-12-15', '2025-01-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)
time_bucket_4 = ('2025-01-15', '2025-02-14', autocallable_notes_chebyshev_approximate, autocallable_note_pricer)


# --------------------------------------------------

p1 = multiprocess.Process(target=create_chebyshev_approximator, args=time_bucket_1)
p2 = multiprocess.Process(target=create_chebyshev_approximator, args=time_bucket_2)
p3 = multiprocess.Process(target=create_chebyshev_approximator, args=time_bucket_3)
p4 = multiprocess.Process(target=create_chebyshev_approximator, args=time_bucket_4)

# starting process 1
p1.start()
# starting process 2
p2.start()
# starting process 3
p3.start()
# starting process 3
p4.start()

# wait until process 1 is finished
p1.join()
# wait until process 2 is finished
p2.join()
p3.join()
p4.join()

# both processes finished
print("Done!")

Done!


In [3]:
'Call_10.mcx'.split(r'_|.')

['Call_10.mcx']